In [1]:
import os, re, json, random, math, textwrap
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import matplotlib.pyplot as plt

# CONFIG 
CONFIG = {
    "FULL_JSONL_PATH": "/kaggle/input/full-original-raw/reddit_pairs_3rd ORIGINAL FULLEST.jsonl",

    "OUT_DIR": "/kaggle/working",

    "PILOT_SIZE_TOTAL": 200,
    "DOUBLE_CODE_SAME_SET": True,

    "RANDOM_SEED": 42,

    "ADD_HELPER_COLUMNS": True,
}

random.seed(CONFIG["RANDOM_SEED"])

# Column schema
ANNOTATION_COLS = [
    # Screening
    "include_pair","exclude_reason",

    # Entities (who talks about whom)
    "speaker_perspective",  # auto/hetero/mixed
    "targets",              # semicolon list (boomer;genx;millennial;genz;other)
    "target_boomer","target_genx","target_millennial","target_genz","target_other","target_other_text",

    # Parent (claim)
    "parent_genericity",    # generic/specific/unclear
    "parent_valence",       # negative/positive/mixed/neutral
    "parent_dim_warmth","parent_dim_competence",   # 0/1
    "Parent text Frame","Parent text Frame (secondary)","parent_topic_other_text",
    # Packaging
    "parent_packaging_noun_label","parent_packaging_adj_label",  # 0/1
    "parent_quantifier","parent_negation","parent_hedge","parent_booster", # 0/1
    "parent_abstraction",   # trait/event/mixed/unclear
    "parent_severity",      # 0/1/2

    # Reply (stance & countermoves)
    "reply_stance",         # support/mitigate/oppose/neutral
    "reply_counter_not_all","reply_counter_counterexample","reply_counter_evidence",
    "reply_counter_alt_cause","reply_counter_reframing","reply_counter_other","reply_counter_other_text",
    "reply_has_claim",

    # If reply has its own claim (mirror)
    "reply_genericity","reply_valence","reply_dim_warmth","reply_dim_competence",
    "reply_topic_primary","reply_topic_secondary",
    "reply_packaging_noun_label","reply_packaging_adj_label",
    "reply_quantifier","reply_negation","reply_hedge","reply_booster",
    "reply_abstraction","reply_severity",

    "notes"
]

# Robust field access (normalize schema variants)
def normalize_row(obj):
    return {
        "pair_id": obj.get("pair_id") or obj.get("id") or obj.get("pairID"),
        "subreddit": obj.get("subreddit"),
        "post_id": obj.get("post_id") or obj.get("link_id") or obj.get("postId"),
        "parent_comment_id": obj.get("parent_comment_id") or obj.get("parent_id"),
        "reply_comment_id": obj.get("reply_comment_id") or obj.get("child_id") or obj.get("reply_id"),

        "post_title": obj.get("post_title") or obj.get("title"),
        "post_selftext": obj.get("post_selftext") or obj.get("selftext"),

        "parent_text": obj.get("parent_text") or obj.get("parent") or obj.get("parentText"),
        "reply_text": obj.get("reply_text") or obj.get("child_text") or obj.get("child") or obj.get("replyText"),
    }

# Target detection for sampling (heuristic)
GEN_PATTERNS = [
    ("boomer", r"\b(baby\s*boomers?|boomers?)\b"),
    ("genx", r"\b(gen\s*[-\s]?x|xennials?)\b"),
    ("millennial", r"\b(millennials?|gen\s*[-\s]?y)\b"),
    ("genz", r"\b(gen\s*[-\s]?z|zoomers?)\b"),
]

def detect_targets(text):
    if not isinstance(text, str):
        return set()
    t = text.lower()
    hits = set()
    for slug, pat in GEN_PATTERNS:
        if re.search(pat, t, flags=re.IGNORECASE):
            hits.add(slug)
    return hits

def auto_targets_for_row(row):
    targets = set()
    for field in ["parent_text", "post_title", "post_selftext"]:
        targets |= detect_targets(row.get(field, ""))
    return sorted(targets)

def load_full_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue
            row = normalize_row(obj)
            rows.append(row)
    df = pd.DataFrame(rows)
    return df

# Balanced sampling (200 by default)
def sample_balanced(df, total=200, seed=42):
    random.seed(seed)
    # add auto target hints (for sampling only)
    auto_t = df.apply(lambda r: ";".join(auto_targets_for_row(r)), axis=1)
    df = df.assign(targets_auto=auto_t)

    groups = ["boomer","genx","millennial","genz"]
    per_group = max(1, total // len(groups))

    # Prefer rows with a single clear target
    idx_single = {g: [] for g in groups}
    idx_multi  = {g: [] for g in groups}

    for i, t in df["targets_auto"].items():
        if not t:
            continue
        ts = t.split(";")
        if len(ts) == 1 and ts[0] in groups:
            idx_single[ts[0]].append(i)
        else:
            for g in ts:
                if g in groups:
                    idx_multi[g].append(i)

    chosen = set()
    picked = []

    # First pass: fill each group from single-target pool
    for g in groups:
        pool = [i for i in idx_single[g] if i not in chosen]
        random.shuffle(pool)
        take = pool[:per_group]
        picked.extend(take)
        chosen.update(take)

    # Second pass: if missing, top-up from multi-target pool
    for g in groups:
        need = per_group - sum((i in chosen) for i in idx_single[g])
        if need <= 0: 
            continue
        pool = [i for i in idx_multi[g] if i not in chosen]
        random.shuffle(pool)
        take = pool[:need]
        picked.extend(take)
        chosen.update(take)

    # If still short, fill from any row that mentions any generation
    if len(picked) < total:
        any_gen = [i for i, t in df["targets_auto"].items() if t and i not in chosen]
        random.shuffle(any_gen)
        take = any_gen[: (total - len(picked))]
        picked.extend(take)
        chosen.update(take)

    picked_unique = []
    seen = set()
    for i in picked:
        if i not in seen:
            picked_unique.append(i)
            seen.add(i)
        if len(picked_unique) >= total:
            break

    return df.loc[picked_unique].copy().reset_index(drop=True)

# Build annotation dataframe
BASE_TEXT_COLS = [
    "pair_id","subreddit","post_id","parent_comment_id","reply_comment_id",
    "post_title","post_selftext","parent_text","reply_text"
]

def add_helper_cols(df):
    def has_link(s): 
        return int(bool(isinstance(s,str) and re.search(r"https?://|www\.", s, re.I)))
    def has_quote(s):
        return int(bool(isinstance(s,str) and re.search(r"(^|\n)>", s)))
    df["parent_char_count"] = df["parent_text"].apply(lambda x: len(x) if isinstance(x,str) else 0)
    df["reply_char_count"] = df["reply_text"].apply(lambda x: len(x) if isinstance(x,str) else 0)
    df["parent_has_link"] = df["parent_text"].apply(has_link)
    df["reply_has_link"] = df["reply_text"].apply(has_link)
    df["parent_has_quote"] = df["parent_text"].apply(has_quote)
    df["reply_has_quote"] = df["reply_text"].apply(has_quote)
    return df

def build_annotation_df(df, add_helpers=True):
    out = df[BASE_TEXT_COLS].copy()
    for col in ANNOTATION_COLS:
        out[col] = ""
    if add_helpers:
        out = add_helper_cols(out)
    return out

# Save coder files
def save_two_coder_files(sample_df, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

    A = build_annotation_df(sample_df, CONFIG["ADD_HELPER_COLUMNS"])
    B = build_annotation_df(sample_df, CONFIG["ADD_HELPER_COLUMNS"])

    # Same 200 for both (double-coding)
    if CONFIG["DOUBLE_CODE_SAME_SET"]:
        A_path = out_dir / "annotation_pilot_coderA.csv"
        B_path = out_dir / "annotation_pilot_coderB.csv"
        A.to_csv(A_path, index=False)
        B.to_csv(B_path, index=False)
        return [str(A_path), str(B_path)]

    # Otherwise, split into two non-overlapping sets of the same size
    n = len(sample_df)
    half = n // 2
    A_df = sample_df.iloc[:half].reset_index(drop=True)
    B_df = sample_df.iloc[half:].reset_index(drop=True)
    A_path = out_dir / "annotation_pilot_coderA.csv"
    B_path = out_dir / "annotation_pilot_coderB.csv"
    build_annotation_df(A_df, CONFIG["ADD_HELPER_COLUMNS"]).to_csv(A_path, index=False)
    build_annotation_df(B_df, CONFIG["ADD_HELPER_COLUMNS"]).to_csv(B_path, index=False)
    return [str(A_path), str(B_path)]

# Lightweight feature detectors (for summary/plots)
QUANTIFIERS = [
    r"all", r"every", r"each", r"always", r"never",
    r"most", r"many", r"some", r"few", r"often",
    r"usually", r"typically", r"generally", r"tend to"
]
HEDGES = [
    r"maybe", r"perhaps", r"seems", r"appears", r"seem to",
    r"i think", r"in my opinion", r"imo", r"i guess", r"i feel",
    r"might", r"could", r"probably", r"likely", r"sort of", r"kind of", r"kinda"
]
BOOSTERS = [
    r"obviously", r"clearly", r"definitely", r"certainly", r"literally",
    r"absolutely", r"undeniably", r"without a doubt", r"no doubt", r"for sure",
    r"everyone knows"
]
NEGATIONS = [
    r"\bnot\b", r"\bnever\b", r"\bno\b", r"\bdon't\b", r"\bdoesn't\b", r"\bdidn't\b",
    r"\bwon't\b", r"\bcan't\b", r"\bcannot\b", r"\bain't\b", r"n['’]t\b"
]

# Packaging (noun vs adjective-like modifier)
NOUN_LABEL = re.compile(r"\b(baby\s*boomers?|boomers?|millennials?|zoomers?|gen\s*[-\s]?z|gen\s*[-\s]?x)\b", re.I)
# adjective-ish: label followed by a non-aux word
AUX_NEXT = {"is","are","was","were","be","being","been","am","do","does","did","have","has","had","can","could","will","would","should","shall","may","might","must"}
ADJ_LABEL = re.compile(r"\b(boomer|millennial|gen\s*[-\s]?z|gen\s*[-\s]?x)\s+([a-z][a-z'-]+)", re.I)

FRAME_KEYWORDS = {
    "work_economy":  ["work","job","jobs","wage","salary","overtime","productivity","retire","retirement","pension"],
    "wealth_housing":["house","housing","rent","mortgage","debt","loan","savings","save","spend","consumption","buy","afford"],
    "tech_media":    ["phone","smartphone","social","tiktok","instagram","youtube","reddit","twitter","gaming","online","internet","news","media"],
    "politics_civics":["vote","voting","policy","policies","rights","protest","election","elections","law","tax","taxes","culture war","civics"],
    "education":     ["school","college","university","degree","diploma","skills","learn","learning","study","studies","teacher","class"],
    "health_aging":  ["health","mental","physical","depress","anxiety","aging","old","older","elderly","decline","fitness","exercise"],
    "family_values": ["family","marriage","married","children","kids","parenting","parents","respect","dating","relationship","values"],
    "environment":   ["climate","environment","emissions","co2","recycling","sustainability","pollution","carbon","warming"],
    "meta_identity": ["generation","generations","gen z","gen x","millennial","boomer","zoomers","label","cohort"]
}

def count_matches(text, patterns):
    if not isinstance(text, str):
        return 0
    t = text.lower()
    c = 0
    for p in patterns:
        # word-boundary-ish, case-insensitive
        if re.search(rf"\b{p}\b", t, flags=re.I):
            c += len(re.findall(rf"\b{p}\b", t, flags=re.I))
    return c

def detect_packaging(text):
    if not isinstance(text, str):
        return (0,0)
    t = text
    noun = 1 if NOUN_LABEL.search(t) else 0
    adj = 0
    # adjective-ish: label + next word isn't an auxiliary
    m = ADJ_LABEL.search(t)
    if m:
        nxt = m.group(2).lower()
        if nxt not in AUX_NEXT:
            adj = 1
    return (noun, adj)

def detect_frame(text):
    if not isinstance(text, str):
        return None
    t = text.lower()
    scores = {}
    for frame, kws in FRAME_KEYWORDS.items():
        s = 0
        for kw in kws:
            s += len(re.findall(rf"\b{re.escape(kw)}\b", t, flags=re.I))
        scores[frame] = s
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else None

def make_summary_and_plots(df_full, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

    # Use PARENT text for claim-like signals
    parents = df_full["parent_text"].fillna("")

    # Counts
    quant_counts = Counter()
    hedge_counts = Counter()
    boost_counts = Counter()
    neg_counts = Counter()

    # Packaging & frames
    noun_count = 0
    adj_count = 0
    frame_counts = Counter()

    for txt in parents:
        for q in QUANTIFIERS: quant_counts[q] += count_matches(txt, [q])
        for h in HEDGES:      hedge_counts[h] += count_matches(txt, [h])
        for b in BOOSTERS:    boost_counts[b] += count_matches(txt, [b])
        neg_counts["negation"] += count_matches(txt, NEGATIONS)

        n, a = detect_packaging(txt)
        noun_count += n
        adj_count  += a
        fr = detect_frame(txt)
        if fr: frame_counts[fr] += 1

    total_rows = len(parents)
    pack_df = pd.DataFrame({
        "feature":["noun_label","adj_label"],
        "count":[noun_count, adj_count],
        "percent":[round(100*noun_count/total_rows,2), round(100*adj_count/total_rows,2)]
    })

    # Save CSV summaries
    def top_n(counter, n=20):
        return pd.DataFrame(counter.most_common(n), columns=["feature","count"])

    qdf = top_n(quant_counts, n=len(quant_counts))
    hdf = top_n(hedge_counts, n=len(hedge_counts))
    bdf = top_n(boost_counts, n=len(boost_counts))
    fdf = top_n(frame_counts, n=len(frame_counts))

    qdf.to_csv(out_dir/"quantifier_counts.csv", index=False)
    hdf.to_csv(out_dir/"hedge_counts.csv", index=False)
    bdf.to_csv(out_dir/"booster_counts.csv", index=False)
    fdf.to_csv(out_dir/"frame_counts.csv", index=False)
    pack_df.to_csv(out_dir/"packaging_counts.csv", index=False)

    # Markdown summary
    md = []
    md.append(f"# Summary (parent_text) — {total_rows} rows\n")
    md.append("## Quantifiers (counts)\n")
    md.append(qdf.to_markdown(index=False))
    md.append("\n\n## Hedges (counts)\n")
    md.append(hdf.to_markdown(index=False))
    md.append("\n\n## Boosters (counts)\n")
    md.append(bdf.to_markdown(index=False))
    md.append("\n\n## Packaging (noun/adj)\n")
    md.append(pack_df.to_markdown(index=False))
    md.append("\n\n## Potential Frames (keyword heuristic)\n")
    md.append(fdf.to_markdown(index=False))
    (out_dir/"summary_stats.md").write_text("\n".join(md), encoding="utf-8")

    # ---- Plots (matplotlib; one plot each; no color specified) ----
    def bar_plot_from_df(df, title, out_png, xlabel="Feature", ylabel="Count"):
        plt.figure(figsize=(10, 5))
        plt.bar(df["feature"], df["count"])
        plt.title(title)
        plt.xlabel(xlabel); plt.ylabel(ylabel)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(out_png, dpi=150)
        plt.close()

    # Show top 15 for readability
    bar_plot_from_df(qdf.head(15), "Top Quantifiers (parent_text)", out_dir/"plot_quantifiers.png")
    bar_plot_from_df(hdf.head(15), "Top Hedges (parent_text)", out_dir/"plot_hedges.png")
    bar_plot_from_df(bdf.head(15), "Top Boosters (parent_text)", out_dir/"plot_boosters.png")
    bar_plot_from_df(fdf,          "Potential Frames (parent_text)", out_dir/"plot_frames.png")
    bar_plot_from_df(pack_df.rename(columns={"feature":"feature","count":"count"}),
                     "Packaging (noun vs adj) — parent_text",
                     out_dir/"plot_packaging.png")

    print("Wrote summaries to:", str(out_dir))

# MAIN
def main():
    full_path = CONFIG["FULL_JSONL_PATH"]
    out_dir = CONFIG["OUT_DIR"]

    assert os.path.exists(full_path), f"Input not found: {full_path}"
    print("Loading:", full_path)
    df_full = load_full_jsonl(full_path)
    print("Loaded rows:", len(df_full))

    # Build balanced pilot
    pilot = sample_balanced(df_full, total=CONFIG["PILOT_SIZE_TOTAL"], seed=CONFIG["RANDOM_SEED"])
    print("Pilot rows:", len(pilot))

    # Save coder files
    paths = save_two_coder_files(pilot, out_dir)
    print("Coder files:", paths)

    # keep a manifest of the sampled pair_ids for reproducibility
    manifest = pilot[["pair_id","subreddit","post_id","parent_comment_id","reply_comment_id"]].copy()
    manifest.to_csv(Path(out_dir)/"pilot_manifest.csv", index=False)

    # Compute summary + plots on the FULL dataset
    make_summary_and_plots(df_full, out_dir)

if __name__ == "__main__":
    main()


Loading: /kaggle/input/full-original-raw/reddit_pairs_3rd ORIGINAL FULLEST.jsonl
Loaded rows: 4199
Pilot rows: 200
Coder files: ['/kaggle/working/annotation_pilot_coderA.csv', '/kaggle/working/annotation_pilot_coderB.csv']
Wrote summaries to: /kaggle/working
